# Notebook 03: Supervised Job Role Classification
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Predict `job_role_label` from unmasked `job_description` (without title).
2. Methodology: Weak supervision context — heuristic regex labels, not human ground truth.
3. Apply stratified 80/10/10 train/val/test split with zero leakage.
4. Classical Baseline: TF-IDF (5,000 n-grams) + Logistic Regression (fit strictly on train).
5. Multilingual SBERT Models:
   - Logistic Regression
   - Linear SVM
   - K-Nearest Neighbors (k=5, cosine metric)
   - Random Forest Classifier
   - XGBoost Classifier
6. Model selection on Validation set; final evaluation on held-out Test set.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.models.classification import run_classification_experiments

## 1. Load Classification Dataset and Embeddings

In [ ]:
df_cls = pd.read_csv(config.PROCESSED_DATA_DIR / "classification_data.csv")
sbert_all = np.load(config.EMBEDDINGS_DIR / "sbert_embeddings.npy")
cleaned_df = pd.read_csv(config.PROCESSED_DATA_DIR / "cleaned_jobs.csv")

id_to_idx = {uid: idx for idx, uid in enumerate(cleaned_df['unique_job_id'])}
cls_indices = [id_to_idx[uid] for uid in df_cls['unique_job_id']]
X_cls = sbert_all[cls_indices]
y_cls = df_cls['job_role_label']

print(f"Classification samples: {X_cls.shape[0]}, Feature dimension: {X_cls.shape[1]}")
print("\nClass Distribution:")
print(y_cls.value_counts())

## 2. Train Models and Execute Benchmark (TF-IDF Baseline + SBERT Models)
This cell executes the actual model training across all algorithms with Stratified 80/10/10 split.

In [ ]:
# Execute live training of TF-IDF baseline and SBERT classifiers
results_df, best_model, meta, split_manifest = run_classification_experiments(
    X_cls,
    y_cls,
    df_cls['job_description'].tolist(),
    df_cls['unique_job_id'].tolist()
)

print("\n=== Classification Models Comparison (Held-Out Test Set) ===")
display(results_df)

## 3. Confusion Matrix and Classification Report

In [ ]:
cr_path = config.REPORTS_DIR / "tables" / "classification_report.csv"
if cr_path.exists():
    cr_df = pd.read_csv(cr_path)
    print("=== Per-Class Classification Report (Best Model on Test Set) ===")
    display(cr_df)

cm_path = config.REPORTS_DIR / "figures" / "classification_confusion_matrix.png"
if cm_path.exists():
    display(Image(filename=str(cm_path)))

comp_path = config.REPORTS_DIR / "figures" / "classification_models_comparison.png"
if comp_path.exists():
    display(Image(filename=str(comp_path)))